In [148]:
import os
import re
import pandas as pd
from matplotlib import pyplot as plt
from tbparse import SummaryReader
from tueplots import bundles

# Keep plotting style consistent with existing slides
plt.rcParams.update(bundles.beamer_moml())
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'figure.dpi': 200})

# change directory to project root
os.chdir(os.path.expanduser("~/Desktop/pomdp_coder"))
print(os.getcwd())

C:\Users\Frederik\Desktop\pomdp_coder


In [167]:
base_dir = os.path.join("outputs")

def load_avg_reward_df(path: str):
    """Return the Average Episode Reward scalars for a given experiment directory."""
    reader = SummaryReader(path, extra_columns={'dir_name'})
    df = reader.scalars
    return df[df['tag'] == "Average Episode Reward"].reset_index(drop=True)

def get_clean_df(path: str):
    """Return a cleaned DataFrame with only relevant columns."""
    df = load_avg_reward_df(path)
    df['environment'] = path.split("\\")[-2]
    df['approach'] = path.split("\\")[-1]
    df['seed'] = df['dir_name'].apply(lambda x: re.search(r"_seed(\d+)", x).group(1))
    df = df.drop(columns=['tag', 'step', 'dir_name'])
    df = df[['environment', 'approach', 'seed', 'value']]
    return df

In [168]:
# add "four_rooms", later, not finished yet
envs = ["tiger", "rocksample", "empty", "corners", "lava", "four_rooms", "unlock"]
methods = ["hardcoded", "ours", "tabular", "random"]

directories = {
    env: {
        method: os.path.join(base_dir, env, method)
        for method in methods
    }
    for env in envs
}
directories

{'tiger': {'hardcoded': 'outputs\\tiger\\hardcoded',
  'ours': 'outputs\\tiger\\ours',
  'tabular': 'outputs\\tiger\\tabular',
  'random': 'outputs\\tiger\\random'},
 'rocksample': {'hardcoded': 'outputs\\rocksample\\hardcoded',
  'ours': 'outputs\\rocksample\\ours',
  'tabular': 'outputs\\rocksample\\tabular',
  'random': 'outputs\\rocksample\\random'},
 'empty': {'hardcoded': 'outputs\\empty\\hardcoded',
  'ours': 'outputs\\empty\\ours',
  'tabular': 'outputs\\empty\\tabular',
  'random': 'outputs\\empty\\random'},
 'corners': {'hardcoded': 'outputs\\corners\\hardcoded',
  'ours': 'outputs\\corners\\ours',
  'tabular': 'outputs\\corners\\tabular',
  'random': 'outputs\\corners\\random'},
 'lava': {'hardcoded': 'outputs\\lava\\hardcoded',
  'ours': 'outputs\\lava\\ours',
  'tabular': 'outputs\\lava\\tabular',
  'random': 'outputs\\lava\\random'},
 'four_rooms': {'hardcoded': 'outputs\\four_rooms\\hardcoded',
  'ours': 'outputs\\four_rooms\\ours',
  'tabular': 'outputs\\four_rooms\\tab

In [169]:
# four_rooms ours did not get logged properly, so we process it separately
path = directories['four_rooms']['ours']
df = SummaryReader(path, extra_columns={'dir_name'}).scalars
df["seed"] = df["dir_name"].apply(lambda s: int(re.search(r'_seed(\d+)', s).group(1)))
reward_df = df[df['tag'] == "Episode Reward"]
avg_ours = reward_df.groupby("seed")["value"].mean().reset_index()
avg_ours["environment"] = "four_rooms"
avg_ours["approach"] = "ours"
avg_ours = avg_ours[["environment", "approach", "seed", "value"]]
extra = pd.DataFrame([{
    "environment": "four_rooms",
    "approach": "ours",
    "seed": 1,
    "value": 0.39883820602832365,
}])

avg_ours = pd.concat([avg_ours, extra], ignore_index=True)
directories['four_rooms'].pop('ours')

'outputs\\four_rooms\\ours'

In [170]:
# four_rooms tabular did not get logged properly, so we process it separately
path = directories['four_rooms']['tabular']
df = SummaryReader(path, extra_columns={'dir_name'}).scalars
df["seed"] = df["dir_name"].apply(lambda s: int(re.search(r'_seed(\d+)', s).group(1)))
reward_df = df[df['tag'] == "Episode Reward"]
avg_tabular = reward_df.groupby("seed")["value"].mean().reset_index()
avg_tabular["environment"] = "four_rooms"
avg_tabular["approach"] = "tabular"
avg_tabular = avg_tabular[["environment", "approach", "seed", "value"]]
directories['four_rooms'].pop('tabular')

'outputs\\four_rooms\\tabular'

In [171]:
# same for random, but all zeros
avg_random = pd.DataFrame([
    {"environment": "four_rooms", "approach": "random", "seed": i, "value": 0.0}
    for i in range(10)
])
directories['four_rooms'].pop('random')

'outputs\\four_rooms\\random'

In [172]:
dfs = []

for env, method in directories.items():
    for approach, dir_path in method.items():
        df = get_clean_df(dir_path)
        dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

In [173]:
# some averages did not get logged, so we add them manually; run only once!
missing_rows = pd.DataFrame([
    {"environment": "lava", "approach": "hardcoded", "seed": 1, "value": 0.06951353308570327},
    {"environment": "lava", "approach": "hardcoded", "seed": 4, "value": 0.12406196502394695},
    {"environment": "lava", "approach": "hardcoded", "seed": 9, "value": 0.25828055561004276},
])

final_df = pd.concat([final_df, avg_ours, avg_tabular, avg_random, missing_rows], ignore_index=True)

In [174]:
# check if wehave 40 for each environment (10 seeds x 4 methods)
final_df["environment"].value_counts()

environment
tiger         40
rocksample    40
empty         40
corners       40
lava          40
four_rooms    40
unlock        40
Name: count, dtype: int64

In [175]:
# write to csv
final_df.to_csv(os.path.join(base_dir, "process_outputs", "baseline_results.csv"), index=False)